# Scoring
In this notebook, we score the BART-filtered headlines for "jitteriness".

First we load the labeled headlines and select only those labeled "relevant" by BART.

In [1]:
import pandas as pd

df = pd.read_csv("data/labeled_headlines.csv")
df = df[df.relevant == 1]
df.dropna(inplace=True)

Then we load the model and define the jitteriness category:

In [2]:
from transformers import pipeline
import torch

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0,
    dtype=torch.float16,
)

jittery = "increasing insecurity"

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Finally, we score each headline:

In [13]:
from datasets import Dataset

ds = Dataset.from_pandas(df[["concat"]])
ds = ds.map(
    lambda x: {"score": classifier(x["concat"], candidate_labels=jittery)},
    batched=True,
    batch_size=128,
)

Map:   0%|          | 0/973 [00:00<?, ? examples/s]

In [ ]:
ds["score"]

Column([[0.9518296122550964], [0.9970592856407166], [0.9418283104896545], [0.8169911503791809], [0.96178138256073], ...])